# Week 7 - Model research & historical replay product

Review saved reports only. **Run All never trains, replays holdout or replaces the production champion.**

Four parameter sets were trained on hosted Kaggle CPU using fit 0-503 and validation 504-671. Repeated validation research is exploratory; fresh unseen chronological data is required before promotion.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

# Find attached completed research output, or local reports when reviewing in the repo.
roots = list(Path('/kaggle/input').rglob('week7_research_decision.json')) if Path('/kaggle/input').exists() else []
REPORTS = roots[0].parent if roots else Path('reports')
if not (REPORTS/'week7_research_decision.json').exists():
    raise FileNotFoundError('Attach the completed Week 7 research outputs')


## 1. Validation decision

The best exploratory score is separate from the frozen deployment champion.


In [ ]:
decision = json.loads((REPORTS/'week7_research_decision.json').read_text())
display(decision)


## 2. Point forecast quality

Lower MAE is better. Compare each horizon against the original C1 validation replay. No holdout metrics appear here.


In [ ]:
benchmark = pd.read_csv(REPORTS/'week7_research_validation.csv')
all_beams = benchmark[benchmark.load_tier == 'All']
mae = all_beams.pivot(index='model',columns='horizon',values='mae')
display(mae)
# Positive percentages mean lower validation MAE than the original C1.
display((100*(1-mae/mae.loc['lightgbm_c1'])).round(2).rename_axis('MAE reduction vs C1 (%)'))


## 3. Paired uncertainty

Difference is candidate MAE minus C1 MAE. Negative favors the candidate. BS bootstrap intervals describe this validation sample, not future performance.


In [ ]:
paired = pd.read_csv(REPORTS/'week7_research_paired.csv')
display(paired[['model','horizon','difference','lower','upper','status']])


## 4. Historical exceedance trade-off

Inspect precision, recall and false-positive rate together. P90/P95 remain fixed from fit data and are not forecast uncertainty quantiles.


In [ ]:
risk = pd.read_csv(REPORTS/'week7_research_risk.csv')
display(risk[['model','horizon','threshold','precision','recall','fpr','positive_events']])


## 5. Load-group diagnostics

Groups come only from the fit segment. Low-load WAPE can be large when its denominator is small.


In [ ]:
display(benchmark.loc[benchmark.load_tier != 'All', ['model','horizon','load_tier','mae','wape','active_only_mae']])


## 6. Working Week 7 product

The FastAPI + Streamlit + Docker product uses the frozen Week 6 champion and 24 real historical beams. From the project root, prepare verified assets and run `docker compose up --build -d --wait`. Open the dashboard at http://127.0.0.1:8501 on the same computer.

The app has no training controls. Its forecasts and risk states come from one shared API. Twenty beam/origin pairs across three horizons matched saved offline predictions at absolute tolerance 1e-9.

Code: [shared predictor](https://github.com/trbk07/5g-network-intelligence/blob/main/src/inference/predictor.py), [API](https://github.com/trbk07/5g-network-intelligence/blob/main/src/api/main.py), [dashboard](https://github.com/trbk07/5g-network-intelligence/blob/main/app/dashboard.py).
